## Project Data Cleaning

### Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import json
import os
from datetime import datetime

In [1]:
import sys
sys.path.append('..')

from utils import calculate_age

In [ ]:
import importlib
import utils
importlib.reload(utils)

In [3]:
# Make plots look nicer
sns.set(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)

# Set paths (adjust if you put data in a subfolder)
DATA_PATH = "../api_data_files"  # update if different

In [4]:
pd.set_option('display.max_columns', None)

### Load in Data

In [5]:
# --- 2. Load JSON files into DataFrames ---
# Example: loop through files if there are multiple
files = os.listdir(DATA_PATH)
print("Files in data folder:", files)

Files in data folder: ['international_box_player_season.json', 'nba_box_player_season.json', 'player.json']


In [6]:
dfs = {}
for file in files:
    if file.endswith(".json"):
        with open(os.path.join(DATA_PATH, file), "r") as f:
            data = json.load(f)
            dfs[file] = pd.DataFrame(data)

In [7]:
dfs.keys()

dict_keys(['international_box_player_season.json', 'nba_box_player_season.json', 'player.json'])

### International Dataset

In [8]:
international_df = dfs['international_box_player_season.json']
international_df.head()

,first_name,last_name,season,season_type,league,team,games,starts,minutes,points,two_points_made,two_points_attempted,three_points_made,three_points_attempted,free_throws_made,free_throws_attempted,blocked_shot_attempts,offensive_rebounds,defensive_rebounds,assists,screen_assists,turnovers,steals,deflections,loose_balls_recovered,blocked_shots,personal_fouls,personal_fouls_drawn,offensive_fouls,charges_drawn,technical_fouls,flagrant_fouls,ejections,points_off_turnovers,points_in_paint,second_chance_points,fast_break_points,possessions,estimated_possessions,team_possessions,usage_percentage,true_shooting_percentage,three_point_attempt_rate,free_throw_rate,offensive_rebounding_percentage,defensive_rebounding_percentage,total_rebounding_percentage,assist_percentage,steal_percentage,block_percentage,turnover_percentage,internal_box_plus_minus
0,theo,greene,2021,Full Season,EuroLeague,Redhawks,25,18,527.61,195,40,78,30,74,25,31,5,8,59,58,0,26,13,0,0,0,40,39,0,0,0,0,0,0,0,0,0,919.7698,919.7698,1778.1345,18.2974,0.5886,0.4868,0.2039,3.2474,13.4519,9.7267,19.0212,1.8221,0.0000,13.5671,0.7786
1,theo,greene,2021,Full Season,Spain - ACB,Redhawks,17,10,356.70,135,26,56,20,43,23,29,2,11,40,37,0,14,7,0,0,1,20,26,0,0,0,0,0,0,0,0,0,652.1629,652.1629,1461.3459,16.9730,0.6040,0.4343,0.2929,3.7781,13.5418,9.9234,14.7139,0.8723,0.8292,11.1323,2.7367
2,miles,brussino,2021,Full Season,EuroCup,Orange,12,2,204.70,82,14,25,11,27,21,22,2,3,27,9,0,10,6,0,0,5,22,20,0,0,0,0,0,0,0,0,0,386.7079,386.7079,1057.9286,16.5449,0.6647,0.5192,0.4231,3.0377,14.5608,10.4102,5.2539,1.5268,3.0675,13.9509,3.2200
3,miles,brussino,2021,Full Season,Italy - Liga A,Orange,16,3,266.00,83,24,47,5,25,20,24,2,7,30,8,0,7,9,0,0,3,24,20,0,0,0,0,0,0,0,0,0,500.4995,500.4995,1354.7356,15.5460,0.5027,0.3472,0.3333,3.7641,14.6187,8.9342,4.4051,1.9879,1.4668,7.8160,-1.5619
4,kadoshnikov,christmas,2012,Full Season,EuroLeague,Redbirds,6,1,70.48,15,3,8,2,15,3,4,1,3,4,3,0,4,2,0,0,2,6,6,0,0,0,0,0,0,0,0,0,129.6115,129.6115,1186.1618,19.0997,0.3029,0.6522,0.1739,4.2915,6.9312,6.6163,5.7849,1.8662,2.4437,13.9082,-5.7589


In [9]:
international_df['season'].value_counts().sort_index()

2010       7
2012     252
2013     213
2014     246
2015     249
2016     264
2017     218
2018     240
2019     257
2020     252
2021    1172
Name: season, dtype: int64

In [10]:
international_df['season_type'].value_counts().sort_index()

Full Season    3370
Name: season_type, dtype: int64

In [11]:
international_df['league'].value_counts().sort_index()

EuroCup            951
EuroLeague        1103
Italy - Liga A     556
Spain - ACB        760
Name: league, dtype: int64

In [12]:
international_df.isna().sum().sort_values(ascending=False)

internal_box_plus_minus            56
free_throw_rate                    49
three_point_attempt_rate           49
true_shooting_percentage           42
turnover_percentage                33
team                               20
possessions                         0
charges_drawn                       0
technical_fouls                     0
flagrant_fouls                      0
ejections                           0
points_off_turnovers                0
points_in_paint                     0
second_chance_points                0
fast_break_points                   0
usage_percentage                    0
estimated_possessions               0
team_possessions                    0
personal_fouls_drawn                0
offensive_rebounding_percentage     0
defensive_rebounding_percentage     0
total_rebounding_percentage         0
assist_percentage                   0
steal_percentage                    0
block_percentage                    0
offensive_fouls                     0
first_name  

In [13]:
international_missing_rows = international_df[international_df[
    ["internal_box_plus_minus", "free_throw_rate", 
     "three_point_attempt_rate", "true_shooting_percentage", 
     "turnover_percentage", "team"]
].isna().any(axis=1)]

Investigating rows that have null values.

In [14]:
international_missing_rows.head(10)

,first_name,last_name,season,season_type,league,team,games,starts,minutes,points,two_points_made,two_points_attempted,three_points_made,three_points_attempted,free_throws_made,free_throws_attempted,blocked_shot_attempts,offensive_rebounds,defensive_rebounds,assists,screen_assists,turnovers,steals,deflections,loose_balls_recovered,blocked_shots,personal_fouls,personal_fouls_drawn,offensive_fouls,charges_drawn,technical_fouls,flagrant_fouls,ejections,points_off_turnovers,points_in_paint,second_chance_points,fast_break_points,possessions,estimated_possessions,team_possessions,usage_percentage,true_shooting_percentage,three_point_attempt_rate,free_throw_rate,offensive_rebounding_percentage,defensive_rebounding_percentage,total_rebounding_percentage,assist_percentage,steal_percentage,block_percentage,turnover_percentage,internal_box_plus_minus
45,zivanovic,pangos,2021,Full Season,EuroLeague,Buffaloes,1,0,0.47,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.8339,0.8339,1657.2596,0.0000,NaN,NaN,NaN,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN
55,zaytsev,bhullar,2021,Full Season,Spain - ACB,Lancers,1,2,2.68,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4.8512,4.8512,1375.7208,0.0000,NaN,NaN,NaN,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN
69,valiev,gordic,2021,Full Season,Spain - ACB,Roadrunners,1,0,7.18,0,0,0,0,0,0,2,0,0,2,1,0,0,2,0,0,0,1,1,0,0,0,0,0,0,0,0,0,13.4412,13.4412,1534.3080,5.7711,0.0000,NaN,NaN,0.0000,33.5826,18.8148,18.5728,14.9082,0.0000,0.0000,NaN
116,norvel,donda,2021,Full Season,Spain - ACB,Lancers,1,0,1.45,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,2.6247,2.6247,1375.7208,0.0000,NaN,NaN,NaN,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN
151,marcelo,mccalebb,2020,Full Season,EuroCup,Terrapins,16,11,437.98,157,17,30,33,94,24,31,3,6,35,26,0,13,11,0,0,1,29,39,0,0,0,0,0,0,0,0,0,774.0804,774.0804,1131.1210,17.4302,0.5703,0.7581,0.2500,1.2773,9.7477,6.9682,12.3903,1.4092,0.0361,8.6298,NaN
153,artest,whaley,2021,Full Season,EuroLeague,Lions,1,0,0.37,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.6406,0.6406,1740.0036,0.0000,NaN,NaN,NaN,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN
156,hite,bochoridis,2021,Full Season,Spain - ACB,Lions,3,0,11.50,2,0,0,0,0,2,2,0,0,4,0,0,3,0,0,0,1,4,1,0,0,0,0,0,0,0,0,0,20.9074,20.9074,1527.8007,16.1386,1.1364,NaN,NaN,0.0000,41.8231,23.9303,0.0000,0.0000,9.8316,77.3196,NaN
167,caicedo,handlogten,2021,Full Season,Italy - Liga A,NaN,15,9,450.00,188,37,74,27,84,33,43,4,11,58,75,0,39,14,0,0,2,37,39,0,0,0,0,0,0,0,0,0,832.1946,832.1946,888.4001,23.2417,0.5310,0.5335,0.2715,2.2926,15.5092,8.3699,32.0013,1.3776,0.0066,18.1252,-1.3095
175,tobey,candi,2021,Full Season,Italy - Liga A,Orange,1,0,1.00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1.8816,1.8816,1354.7356,0.0000,NaN,NaN,NaN,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN
204,zanotti,abalde,2021,Full Season,Italy - Liga A,NaN,12,5,355.00,137,39,72,14,47,17,22,5,6,29,20,0,20,13,0,0,1,30,27,0,0,0,0,0,0,0,0,0,654.4821,654.4821,833.1217,20.1227,0.5317,0.3897,0.1763,1.0336,9.0078,6.2369,9.8850,1.8760,0.1204,13.7474,-4.7616


In [15]:
international_df[(international_df['first_name'] == 'zanna') & (international_df['last_name'] == 'bairstow')]

,first_name,last_name,season,season_type,league,team,games,starts,minutes,points,two_points_made,two_points_attempted,three_points_made,three_points_attempted,free_throws_made,free_throws_attempted,blocked_shot_attempts,offensive_rebounds,defensive_rebounds,assists,screen_assists,turnovers,steals,deflections,loose_balls_recovered,blocked_shots,personal_fouls,personal_fouls_drawn,offensive_fouls,charges_drawn,technical_fouls,flagrant_fouls,ejections,points_off_turnovers,points_in_paint,second_chance_points,fast_break_points,possessions,estimated_possessions,team_possessions,usage_percentage,true_shooting_percentage,three_point_attempt_rate,free_throw_rate,offensive_rebounding_percentage,defensive_rebounding_percentage,total_rebounding_percentage,assist_percentage,steal_percentage,block_percentage,turnover_percentage,internal_box_plus_minus
827,zanna,bairstow,2014,Full Season,Italy - Liga A,Paladins,32,32,934.00,550,122,240,57,156,135,195,13,22,97,99,0,103,50,0,0,1,95,203,0,0,0,0,0,0,0,0,0,1697.0356,1697.0356,2407.4648,29.9016,0.5708,0.3939,0.4924,3.5520,13.6685,7.5027,19.8224,2.9226,0.6522,17.6129,2.8856
828,zanna,bairstow,2015,Full Season,EuroCup,Seminoles,6,4,154.28,86,20,41,9,32,19,23,3,3,14,25,0,17,5,0,0,0,15,34,0,0,0,0,0,0,0,0,0,287.7131,287.7131,447.5591,30.3855,0.5173,0.4384,0.3151,2.8957,13.6334,7.6904,31.0806,1.7780,0.0000,16.9796,-0.7129
829,zanna,bairstow,2015,Full Season,EuroLeague,Seminoles,10,10,238.59,102,23,57,13,45,17,32,7,9,16,30,0,34,17,0,0,1,29,35,0,0,0,0,0,0,0,0,0,447.2983,447.2983,749.9029,28.4243,0.4394,0.4412,0.3137,4.2716,8.6217,7.1705,22.6715,3.5601,0.5816,22.6546,-4.9372
830,zanna,bairstow,2015,Full Season,Italy - Liga A,Seminoles,47,45,1376.00,698,147,336,71,276,191,243,27,44,138,190,0,151,96,0,0,2,147,275,0,0,0,0,0,0,0,0,0,2559.6941,2559.6941,3646.0759,28.3494,0.4855,0.4510,0.3971,4.9954,14.0750,6.9038,27.0019,3.4926,0.1025,17.3579,3.0114
831,zanna,bairstow,2016,Full Season,Italy - Liga A,Dukes,18,17,546.00,285,50,115,42,124,59,84,9,14,38,56,0,52,31,0,0,1,50,104,0,0,0,0,0,0,0,0,0,1012.4885,1012.4885,2253.0651,28.7038,0.5164,0.5188,0.3515,4.5397,10.7995,5.4079,17.2825,2.9330,0.0274,15.8556,2.3437
832,zanna,bairstow,2017,Full Season,EuroCup,Eagles,19,11,498.11,209,36,78,31,87,44,59,11,6,47,42,0,50,21,0,0,2,52,77,0,0,0,0,0,0,0,0,0,899.2997,899.2997,1372.1367,23.6196,0.5472,0.5273,0.3576,1.5575,11.0892,6.5404,13.8958,2.0573,0.6089,20.7503,-0.6188
833,zanna,bairstow,2018,Full Season,EuroCup,Eagles,10,10,313.39,157,28,57,20,52,41,52,5,5,28,41,0,28,17,0,0,0,31,53,0,0,0,0,0,0,0,0,0,582.1749,582.1749,743.0602,24.2817,0.5952,0.4771,0.4771,2.1594,14.1652,5.7491,25.3073,3.1984,0.0000,17.5131,1.8161
834,zanna,bairstow,2020,Full Season,Italy - Liga A,Longhorns,20,18,597.00,260,64,132,26,119,54,68,6,14,52,95,0,67,26,0,0,1,56,106,0,0,0,0,0,0,0,0,0,1111.7447,1111.7447,1797.0413,27.7479,0.4628,0.4741,0.2709,2.8596,11.6484,7.0856,24.7878,2.8835,0.0275,19.2573,-4.1364


So for players that have multiple rows in the same season, it is because they played in multiple leagues. Based on an earlier cell, I know I have enough data points for each league so I don't need to combine these players into one row. I can perform separate analyses of how each league translates to the NBA. This will help me account for any differences in competition level.

**Cleaning plan:**
- For free throw rate, three point attempt rate, true shooting percentage, and turnover percentage, the null values seem to come when the data should be 0.
- For team, this is irrelevant so I will leave the data as is.
- For the internal box plus minus, I will look at the distribution by league to see if I should assign a 0 or a median number.

In [16]:
ibpm_stats = international_df.groupby("league")["internal_box_plus_minus"].agg(["count", "mean", "median", "std", "min", "max"])
print(ibpm_stats)

                count      mean   median       std      min      max
league                                                              
EuroCup           935 -0.792606 -0.11880  6.280922 -40.6780  28.7695
EuroLeague       1088 -0.715041  0.17865  6.382560 -67.8965  38.2655
Italy - Liga A    546 -0.682442 -0.27910  5.442392 -54.5786  27.0794
Spain - ACB       745 -0.683135 -0.15470  5.357958 -42.9841  13.3787


In [17]:
ibpm_stats = international_df.groupby("season")["internal_box_plus_minus"].agg(["count", "mean", "median", "std", "min", "max"])
print(ibpm_stats)

        count      mean   median       std      min      max
season                                                      
2010        7  0.287414 -3.21710  6.711284  -5.7367  11.6516
2012      251 -0.409537 -0.05310  5.521695 -32.0258  14.8601
2013      213  0.344798  0.71760  5.046025 -24.2969  25.6504
2014      246  0.167484  0.95115  5.508006 -40.6780  12.7943
2015      248 -0.443698  0.31290  6.709340 -67.8965  11.1294
2016      264  0.014928  0.51045  4.694901 -23.4041  12.1957
2017      218 -0.195824  0.14755  4.378662 -18.5120  11.9548
2018      240 -0.308414  0.48555  4.810307 -20.2356  11.7101
2019      257 -0.269650  0.01480  4.549456 -15.7890  11.0333
2020      245 -0.189884  0.46950  5.778444 -59.0112  12.1896
2021     1125 -1.845187 -0.88240  7.032321 -54.5786  38.2655


For imputing data in the internal box plus minus column, I checked the median on two different splits: season and league. The split by league seems to have more consistent and reliable numbers, so I will impute the league median for the 56 rows with null values for internal box plus minus.

In [18]:
# Impute internal_box_plus_minus with league-specific medians
international_df['internal_box_plus_minus'] = international_df.groupby('league')['internal_box_plus_minus']\
    .transform(lambda x: x.fillna(x.median()))

Imputing null values to 0.

In [19]:
cols_to_zero = [
    "free_throw_rate", 
    "three_point_attempt_rate", 
    "true_shooting_percentage", 
    "turnover_percentage"
]

international_df[cols_to_zero] = international_df[cols_to_zero].fillna(0)

Checked that now there are only null values in the team column. Removed that code here to shorten the notebook.

### NBA Dataset

In [21]:
nba_df = dfs['nba_box_player_season.json']
nba_df.head()

,first_name,last_name,season,season_type,league,team,games,starts,minutes,points,plus_minus,two_points_made,two_points_attempted,three_points_made,three_points_attempted,free_throws_made,free_throws_attempted,blocked_shot_attempts,offensive_rebounds,defensive_rebounds,assists,screen_assists,turnovers,steals,deflections,loose_balls_recovered,blocked_shots,personal_fouls,personal_fouls_drawn,offensive_fouls,charges_drawn,technical_fouls,flagrant_fouls,ejections,points_off_turnovers,points_in_paint,second_chance_points,fast_break_points,possessions,estimated_possessions,calculated_possessions,plays_used,team_possessions,usage_percentage,true_shooting_percentage,three_point_attempt_rate,free_throw_rate,offensive_rebounding_percentage,defensive_rebounding_percentage,total_rebounding_percentage,assist_percentage,steal_percentage,block_percentage,turnover_percentage,internal_box_plus_minus
0,Kadoshnikov,Christmas,2017,Full Season,NBA,Thunder,73,6,1135.2833,430,-49,43,100,99,264,47,53,9,20,75,44,14,33,37,62,22,8,122,0,0,1,0,0,0,0,0,0,0,2504.5,2314.6044,2504.5,421.0,8564.9572,15.7279,0.5551,0.7253,0.1456,2.2515,8.3138,4.1831,7.9835,1.7237,0.6262,7.8512,-1.8309
1,Kadoshnikov,Christmas,2018,Full Season,NBA,Thunder,81,8,1243.8000,377,69,33,77,90,234,41,48,11,29,100,30,4,26,43,78,47,10,135,0,0,2,0,0,0,0,0,0,0,2666.0,2507.9530,2666.0,361.0,8577.6105,12.3717,0.5676,0.7524,0.1543,2.7414,10.2934,6.3177,5.7126,1.7110,1.1719,7.2601,-1.3166
2,Kadoshnikov,Christmas,2019,Full Season,NBA,Thunder,31,2,588.1667,165,29,15,30,41,127,12,13,6,5,43,20,1,14,17,29,17,6,53,0,0,2,1,0,0,0,0,0,0,1272.0,1258.1393,1272.0,165.0,9007.6928,12.2666,0.5070,0.8089,0.0828,0.9124,7.2142,4.8119,3.7177,1.5495,1.2268,7.9221,-3.1117
3,Kurucs,Humphrey,2013,Full Season,NBA,Raptors,29,0,342.3500,116,96,41,73,1,2,31,38,5,30,47,11,0,17,13,0,0,15,53,0,0,0,5,1,0,0,0,0,0,678.5,644.7166,678.5,112.0,7525.3031,14.7308,0.6324,0.0267,0.5067,11.5375,18.4324,14.6673,6.3495,2.0357,3.8387,15.6365,0.0371
4,Kurucs,Humphrey,2014,Full Season,NBA,NaN,63,0,847.7000,171,-140,62,126,4,15,35,53,10,72,144,28,0,30,23,0,0,26,122,0,0,0,5,0,0,0,0,0,0,1754.5,1662.7473,1754.5,196.0,7829.1455,10.1367,0.5195,0.0989,0.3701,11.0518,18.4784,14.5251,4.4947,1.5469,2.8704,15.6001,-1.7587


In [22]:
nba_df['season'].value_counts().sort_index()

2010    137
2011    141
2012    162
2013    158
2014    175
2015    171
2016    157
2017    153
2018    165
2019    116
2020     84
2021     66
Name: season, dtype: int64

In [23]:
nba_df['season_type'].value_counts().sort_index()

Full Season    1685
Name: season_type, dtype: int64

In [24]:
nba_df.isna().sum().sort_values(ascending=False)

team                               246
free_throw_rate                     11
true_shooting_percentage            11
three_point_attempt_rate            11
turnover_percentage                  9
calculated_possessions               3
plays_used                           3
internal_box_plus_minus              1
possessions                          0
points_in_paint                      0
ejections                            0
flagrant_fouls                       0
technical_fouls                      0
second_chance_points                 0
fast_break_points                    0
charges_drawn                        0
points_off_turnovers                 0
team_possessions                     0
estimated_possessions                0
personal_fouls_drawn                 0
usage_percentage                     0
offensive_rebounding_percentage      0
defensive_rebounding_percentage      0
total_rebounding_percentage          0
assist_percentage                    0
steal_percentage         

In [25]:
nba_missing_rows = nba_df[nba_df[
    ["internal_box_plus_minus", "free_throw_rate", 
     "three_point_attempt_rate", "true_shooting_percentage", 
     "turnover_percentage", "calculated_possessions", "plays_used"]
].isna().any(axis=1)]

Once again, I don't care about the missing values for the team.

In [26]:
nba_missing_rows

,first_name,last_name,season,season_type,league,team,games,starts,minutes,points,plus_minus,two_points_made,two_points_attempted,three_points_made,three_points_attempted,free_throws_made,free_throws_attempted,blocked_shot_attempts,offensive_rebounds,defensive_rebounds,assists,screen_assists,turnovers,steals,deflections,loose_balls_recovered,blocked_shots,personal_fouls,personal_fouls_drawn,offensive_fouls,charges_drawn,technical_fouls,flagrant_fouls,ejections,points_off_turnovers,points_in_paint,second_chance_points,fast_break_points,possessions,estimated_possessions,calculated_possessions,plays_used,team_possessions,usage_percentage,true_shooting_percentage,three_point_attempt_rate,free_throw_rate,offensive_rebounding_percentage,defensive_rebounding_percentage,total_rebounding_percentage,assist_percentage,steal_percentage,block_percentage,turnover_percentage,internal_box_plus_minus
54,Elton,Bagaric,2013,Full Season,NBA,Lakers,1,0,33.4700,10,0,3,6,1,7,1,1,1,2,4,0,0,1,0,0,0,2,6,0,0,0,0,0,0,0,0,0,0,65.7645,65.7645,NaN,NaN,8140.4974,19.2429,0.3720,0.5385,0.0769,8.0116,14.5012,9.5015,0.0000,0.0000,4.7140,6.9252,NaN
152,Davidson,Ndoye,2015,Full Season,NBA,Jazz,2,0,2.7333,0,6,0,0,0,0,0,0,0,1,2,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6.0000,5.1474,6.0,1.0,7421.6467,16.7718,NaN,NaN,NaN,42.6852,83.6387,62.6683,0.0000,0.0000,0.0000,100.0000,-11.9618
326,Imbro,Gasol,2014,Full Season,NBA,Pelicans,4,0,23.8500,0,12,0,0,0,0,0,0,0,0,3,2,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,47.5000,45.7987,47.5,1.0,7625.4324,1.9046,NaN,NaN,NaN,0.0000,15.5489,7.2101,9.4750,2.2201,0.0000,100.0000,-2.0022
378,Gortat,Henderson,2014,Full Season,NBA,Pistons,34,0,237.5333,83,37,27,58,7,39,8,10,0,14,32,11,0,11,6,0,0,1,20,0,0,0,0,0,0,0,0,0,0,469.7557,469.7557,NaN,NaN,7823.5478,20.2466,0.4093,0.4021,0.1031,5.2231,14.8061,10.7138,8.2213,1.1133,0.5255,9.7865,-6.0860
379,Gortat,Henderson,2015,Full Season,NBA,NaN,24,1,222.8833,109,69,27,53,18,42,1,1,0,3,27,10,0,9,2,0,0,7,19,0,0,0,0,0,0,0,0,0,0,443.4501,443.4501,NaN,NaN,8262.0722,20.5647,0.5771,0.4481,0.0111,2.0893,15.1523,8.8660,7.2859,0.2564,2.5806,8.2529,0.3473
393,Voigtmann,Kljajic,2016,Full Season,NBA,Rockets,3,0,6.1500,0,-2,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,16.0000,12.5144,16.0,0.0,8558.5975,0.0000,NaN,NaN,NaN,0.0000,18.2457,8.4783,0.0000,8.0576,0.0000,NaN,3.6378
672,Rodijs,Kurbanov,2019,Full Season,NBA,Cavaliers,1,0,0.7333,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1.5000,1.4765,1.5,0.0,7954.9980,0.0000,NaN,NaN,NaN,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,-3.6749
738,Norense,Galloway,2013,Full Season,NBA,Nets,2,0,0.3833,0,-1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2.5000,0.7085,2.5,0.0,7988.0485,0.0000,NaN,NaN,NaN,0.0000,98.8222,99.7769,0.0000,0.0000,0.0000,NaN,16.4979
802,Octavius,Smailagic,2013,Full Season,NBA,Warriors,2,0,1.9000,0,-6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6.0000,3.7369,6.0,0.0,8952.8256,0.0000,NaN,NaN,NaN,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,-1.6193
840,Pickard,Shaw Jr.,2019,Full Season,NBA,Suns,1,0,5.9000,0,-9,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,12.5000,12.3562,12.5,0.0,8326.8264,0.0000,NaN,NaN,NaN,0.0000,19.1579,8.6804,0.0000,0.0000,0.0000,NaN,-6.6616


In [27]:
player_season_counts = (
    nba_df.groupby(['first_name', 'last_name', 'season'])
    .size()
    .reset_index(name='row_count')
)

# Show only those with more than 1 row
player_season_counts[player_season_counts['row_count'] > 1]

,first_name,last_name,season,row_count


Each player only has one season, so I don't need code of combining rows into a total row if a player played on multiple teams.

**Cleaning plan:**
- For free throw rate, three point attempt rate, true shooting percentage, and turnover percentage, the null values seem to come when the data should be 0.
- For team, this is irrelevant so I will leave the data as is.
- For the internal box plus minus, I will input the median.
- For calculated possessions and plays used, I can compute it using usage percentage, estimated possessions. 
**Plays used / Calculated Possessions = Usage Percentage.** And I can get calculated possessions by calculating the relationship between calculated possessions and estimated possessions.

First starting with the rows to impute to 0.

In [28]:
cols_to_zero = [
    "free_throw_rate", 
    "three_point_attempt_rate", 
    "true_shooting_percentage", 
    "turnover_percentage"
]

nba_df[cols_to_zero] = nba_df[cols_to_zero].fillna(0)

In [29]:
# Only keep rows with non-null IBPM and non-zero plus_minus
df_non_null = nba_df.dropna(subset=['internal_box_plus_minus', 'plus_minus'])
df_non_null = df_non_null[df_non_null['plus_minus'] != 0]

# Correlation
corr = df_non_null['internal_box_plus_minus'].corr(df_non_null['plus_minus'])
print(f"Correlation between IBPM and plus_minus (non-zero only): {corr:.3f}")

# Ratio
ratio = (df_non_null['internal_box_plus_minus'] / df_non_null['plus_minus']).mean()
print(f"Average ratio (IBPM / plus_minus): {ratio:.3f}")

Correlation between IBPM and plus_minus (non-zero only): 0.196
Average ratio (IBPM / plus_minus): 0.159


I wanted to check if there was a relationship between plus minus and internal box plus minus to use that as a way to impute data instead. However, the correlation is not strong, so I will just use the median again.

In [30]:
ibpm_stats = nba_df["internal_box_plus_minus"].agg(["count", "mean", "median", "std", "min", "max"])
print(ibpm_stats)

count     1684.000000
mean        -2.474006
median      -2.028200
std          4.700979
min        -54.395300
max         39.988400
Name: internal_box_plus_minus, dtype: float64


In [31]:
# Impute internal_box_plus_minus with overall dataset median
nba_df['internal_box_plus_minus'] = nba_df['internal_box_plus_minus']\
    .fillna(nba_df['internal_box_plus_minus'].median())

Next, working on calculated possessions and plays used.

In [32]:
# Only keep rows with non-null calculated possessions and estimated possessions and non-zero plus_minus
df_non_null = nba_df.dropna(subset=['calculated_possessions', 'estimated_possessions'])
df_non_null = df_non_null[df_non_null['estimated_possessions'] != 0]

# Correlation
corr = df_non_null['calculated_possessions'].corr(df_non_null['estimated_possessions'])
print(f"Correlation between Calculated possessions and estimated possessions (non-zero only): {corr:.3f}")

# Ratio
ratio = (df_non_null['calculated_possessions'] / df_non_null['estimated_possessions']).mean()
print(f"Average ratio (Calculated possessions / estimated possessions): {ratio:.3f}")

Correlation between Calculated possessions and estimated possessions (non-zero only): 0.999
Average ratio (Calculated possessions / estimated possessions): 1.069


In [33]:
# Your computed ratio
ratio = 1.069  

# Fill missing calculated_possessions using estimated_possessions * ratio
nba_df['calculated_possessions'] = nba_df['calculated_possessions'].fillna(
    nba_df['estimated_possessions'] * ratio
)

In [34]:
# Impute missing plays_used as calculated_possessions * usage_percentage
nba_df['plays_used'] = nba_df['plays_used'].fillna(
    nba_df['calculated_possessions'] * nba_df['usage_percentage']
)

Checked that now there are only null values in the team column. Removed that code here to shorten the notebook.

### Player Dictionary

In [36]:
player_df = dfs['player.json']
player_df.head()

,first_name,last_name,birth_date
0,theo,greene,1995-12-26
1,miles,brussino,1993-02-01
2,ayres,bortolani,1969-01-20
3,kadoshnikov,christmas,1993-08-10
4,rashawn,de,1989-08-30


In [37]:
player_df.isna().sum().sort_values(ascending=False)

first_name    0
last_name     0
birth_date    0
dtype: int64

Nothing that needs to be cleaned here.

### Exporting Cleaned Data

In [38]:
international_df['full_name'] = international_df['first_name'] + " " + international_df['last_name']
nba_df['full_name'] = (nba_df['first_name'].str.lower() + " " + nba_df['last_name'].str.lower())
player_df['full_name'] = player_df['first_name'] + " " + player_df['last_name']

In [39]:
nba_df['full_name'] = nba_df['full_name'].str.replace(
    r'\s+(jr\.|iii|iv)$', '', regex=True, case=False
)

In order for me to join the datasets later I need to standardize the names. In going through the examples, the international and player datasets already match. I just needed to make some edits to the NBA dataset for 'full_name' to match the other datasets. The changes include making the name lower case and removing any suffixes such as Jr., III, or IV.

In [40]:
# --- Step 1: Merge player_df (only birth_date + full_name) with international_df and nba_df ---
player_birth = player_df[['full_name', 'birth_date']].copy()

# Merge on full_name (left join)
international_df = international_df.merge(player_birth, on='full_name', how='left')
nba_df = nba_df.merge(player_birth, on='full_name', how='left')


In [41]:
# --- Step 2: Convert birth_date to datetime if not already ---
international_df['birth_date'] = pd.to_datetime(international_df['birth_date'], errors='coerce')
nba_df['birth_date'] = pd.to_datetime(nba_df['birth_date'], errors='coerce')

In [43]:
international_df['age'] = international_df.apply(calculate_age, axis=1)
nba_df['age'] = nba_df.apply(calculate_age, axis=1)

For the two datasets, I did an initial join to the player dictionary to be able to take in birth date and derive an age column. The four leagues in the international dataset end around June, so I assumed the date to be July 1 for whatever year the season just ended and used that to calculate age.

In [108]:
player_df.to_csv("../outputs/cleaned_player_df.csv", index=False)
international_df.to_csv("../outputs/cleaned_international_df.csv", index=False)
nba_df.to_csv("../outputs/cleaned_nba_df.csv", index=False)

x